# Demonstrating parallel simcat database simulation on HPC with SLURM

### Imports

In [ ]:
import toytree
import simcat
import subprocess
import os
import h5py
import numpy as np

### Define the species tree model and database params

In [ ]:
tre = toytree.rtree.imbtree(4,6e6)
tre.draw(ts='p');

In [ ]:
num_rows_db = 500 # number of simulations in the database
db = simcat.Database("hpc_demo",
                '../dbs/',
                tre,
                nrows=num_rows_db,
                nsnps=200,
                Ne_min=10000, # how much should Ne vary on the branches?
                Ne_max=20000,
                admix_prop_min=0.1, # how much should the magnitude of admixture event vary?
                admix_prop_max=0.3,
                admix_edge_min=0.1, # how much should the timing of admixture event vary?
                admix_edge_max=0.9,
                exclude_sisters=True, # do we want to include introgression between sister taxa?
                node_slide_prop=0.25, # how much do we want internal nodes to shift around?
                existing_admix_edges=[],
                    ) # do we want to assume any existing edges?

### Write out base scripts

#### simcat `simulate` script: inits the simulator based on the database name, specifies the number of simulations per job

In [ ]:
database_dir = os.path.abspath("../dbs")
simulate_script = f"""import simcat
simulator = simcat.Simulator("hpc_demo", {database_dir!r})  # inits the simulator
simulator.run(20,auto=True) # runs as many simulations as we specify, automatically detects available cores
"""
simulate_script_path = "../scripts/run_simcat_HPC_demo.py"
os.makedirs(os.path.dirname(simulate_script_path), exist_ok=True)

In [ ]:
with open(simulate_script_path,'w') as f:
    f.write(simulate_script)

#### SLURM template: specify the length of number of cores per job, length of each job, memory, partition, etc. 

#### These parameters could be tuned by running jobs with few simulations and examining efficiency afterward with `seff {job_id}`

In [ ]:
slurm_template = """#!/bin/bash
#SBATCH -c 4
#SBATCH -t 0-3:59:00
#SBATCH -p shared
#SBATCH --mem-per-cpu=2G
#SBATCH --job-name=hpc_{job_num}
#SBATCH -o {log_dir}/hpc_{job_num}.out
#SBATCH -e {log_dir}/hpc_{job_num}.err

# Activate the environment where simcat is installed. Replace this
# placeholder with your cluster's activation command.
source /path/to/conda-or-mamba/bin/activate simcat-env

# Deactivate file locking, we do this manually
export HDF5_USE_FILE_LOCKING=FALSE

# Navigate to script directory
cd {script_dir}

# Execute the Python script
python run_simcat_HPC_demo.py
"""

In [ ]:
# specify the directory to save the slurm scripts in (one submit script per job, plus .err and .out files as they run)
slurm_script_dir = "../scripts/slurm_simulate_scripts"
os.makedirs(slurm_script_dir, exist_ok=True)

# absolute paths to script and log directories
abs_script_dir = os.path.abspath(os.path.dirname(simulate_script_path))
abs_log_dir = os.path.abspath(slurm_script_dir)

### Run SLURM jobs in parallel to fill the database with simulations

In [ ]:
# how many jobs needed?
sims_per_job = 20 # taken from `run_simcat_HPC_demo.py` script above
num_rows_db / sims_per_job

In [ ]:
num_jobs = 25  # change this to number of jobs

# submit jobs in loop
for i in range(num_jobs):
    slurm_script_content = slurm_template.format(
        job_num=i,
        script_dir=abs_script_dir,
        log_dir=abs_log_dir
    )
    slurm_script_file = os.path.join(slurm_script_dir, f"hpc_{i}.sh")

    # write SLURM script to file
    with open(slurm_script_file, "w") as f:
        f.write(slurm_script_content)

    # submit to SLURM
    subprocess.run(["sbatch", slurm_script_file], check=True)

### Once the jobs finish running: Confirm that all jobs completed successfully 
#### -> Check the `finished_sims` dataset in the labels file. Finished jobs will have a value of 1.

In [ ]:
# open labels file in read mode
with h5py.File('../dbs/hpc_demo.labels.h5', 'r') as labs_file:
    # access the finished_sims dataset
    finished_data = labs_file['finished_sims']

    # how many indices currently marked as 1 (= finished)?
    print(np.sum(finished_data[:] == 1))

#### You can also use `sacct` and `seff` to examine outcomes of individual jobs